# 🥇 Gold Layer — primeinsurance
## All 7 Tables

| Table | Type | Source |
|---|---|---|
| `fact_claims` | Fact | silver.claims + silver.policy + silver.customers |
| `fact_sales` | Fact | silver.sales + silver.cars |
| `dim_customer` | Dimension | silver.customers |
| `dim_policy` | Dimension | silver.policy |
| `dim_car` | Dimension | silver.cars |
| `dim_region` | Dimension | silver.customers + silver.sales |
| `dim_date` | Dimension | silver.sales (ad_placed_on + sold_on) |


## Setup

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS primeinsurance.gold")
print("primeinsurance.gold schema ready")


## fact_claims
`silver.claims` + `silver.policy` + `silver.customers`

Join: `claims.PolicyID → policy.policy_number → customers.customer_id`

In [0]:
%sql
select * from primeinsurance.silver.sales

In [0]:
from pyspark.sql.functions import col, current_timestamp

claims    = spark.table("primeinsurance.silver.claims")
policy    = spark.table("primeinsurance.silver.policy")
customers = spark.table("primeinsurance.silver.customers")

fact_claims = (
    claims
    .join(policy,    claims["PolicyID"] == policy["policy_number"], "left")
    .join(customers, "customer_id", "left")
    .select(
        # Keys
        col("ClaimID").alias("claim_id"),
        col("PolicyID").alias("policy_id"),
        col("customer_id"),
        col("car_id"),

        # Incident
        col("incident_date"),
        col("incident_type"),
        col("incident_severity"),
        col("collision_type"),
        col("incident_state"),
        col("incident_city"),
        col("incident_location"),
        col("authorities_contacted"),
        col("number_of_vehicles_involved"),

        # Financials
        col("injury"),
        col("property"),
        col("vehicle"),
        col("bodily_injuries"),
        col("witnesses"),

        # Flags
        col("property_damage"),
        col("police_report_available"),
        col("Claim_Rejected").alias("claim_rejected"),
        col("Claim_Logged_On").alias("claim_logged_on"),
        col("Claim_Processed_On").alias("claim_processed_on"),

        # Policy
        col("policy_bind_date"),
        col("policy_state"),
        col("policy_csl"),
        col("policy_deductable"),
        col("policy_annual_premium"),
        col("umbrella_limit"),

        # Customer
        col("region"),
        customers["state"].alias("customer_state"),
        customers["city"].alias("customer_city"),
        col("job"),
        col("marital_status"),
        col("education"),
        col("default"),
        col("balance"),
        col("hh_insurance"),
        col("car_loan"),

        # Lineage
        claims["source_file"],
        claims["load_timestamp"],
        current_timestamp().alias("gold_created_at")
    )
)

(
    fact_claims.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.fact_claims")
)

print(f" gold.fact_claims  : {fact_claims.count():,} rows, {len(fact_claims.columns)} columns")


## fact_sales
`silver.sales` + `silver.cars`

Join: `sales.car_id → cars.car_id`

In [0]:
from pyspark.sql.functions import col, current_timestamp

sales = spark.table("primeinsurance.silver.sales")
cars  = spark.table("primeinsurance.silver.cars")

fact_sales = (
    sales
    .join(cars, "car_id", "left")
    .select(
        # Keys
        col("sales_id"),
        col("car_id"),

        # Sale info
        col("ad_placed_on"),
        col("sold_on"),
        col("original_selling_price"),
        col("Region"),
        col("State"),
        col("City"),
        col("seller_type"),
        col("owner"),

        # Car info
        col("name").alias("car_name"),
        col("model"),
        col("fuel"),
        col("transmission"),
        col("km_driven"),
        col("mileage"),
        col("engine"),
        col("max_power"),
        col("torque"),
        col("seats"),

        # Lineage
        sales["source_file"],
        sales["load_timestamp"],
        current_timestamp().alias("gold_created_at")
    )
)

(
    fact_sales.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.fact_sales")
)

print(f" gold.fact_sales   : {fact_sales.count():,} rows, {len(fact_sales.columns)} columns")


## dim_customer
`silver.customers`

In [0]:
from pyspark.sql.functions import current_timestamp

customers = spark.table("primeinsurance.silver.customers")

dim_customer = (
    customers
    .select(
        "customer_id",
        "region",
        "state",
        "city",
        "job",
        "marital_status",
        "education",
        "default",
        "balance",
        "hh_insurance",
        "car_loan",
        "source_file",
        "load_timestamp",
        current_timestamp().alias("gold_created_at")
    )
)

(
    dim_customer.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.dim_customer")
)

print(f" gold.dim_customer : {dim_customer.count():,} rows, {len(dim_customer.columns)} columns")


## dim_policy
`silver.policy`

In [0]:
from pyspark.sql.functions import current_timestamp

policy = spark.table("primeinsurance.silver.policy")

dim_policy = (
    policy
    .select(
        "policy_number",
        "customer_id",
        "car_id",
        "policy_bind_date",
        "policy_state",
        "policy_csl",
        "policy_deductable",
        "policy_annual_premium",
        "umbrella_limit",
        "source_file",
        "load_timestamp",
        current_timestamp().alias("gold_created_at")
    )
)

(
    dim_policy.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.dim_policy")
)

print(f" gold.dim_policy   : {dim_policy.count():,} rows, {len(dim_policy.columns)} columns")


## dim_car
`silver.cars`

In [0]:
from pyspark.sql.functions import current_timestamp

cars = spark.table("primeinsurance.silver.cars")

dim_car = (
    cars
    .select(
        "car_id",
        "name",
        "model",
        "fuel",
        "transmission",
        "km_driven",
        "mileage",
        "engine",
        "max_power",
        "torque",
        "seats",
        "source_file",
        "load_timestamp",
        current_timestamp().alias("gold_created_at")
    )
)

(
    dim_car.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.dim_car")
)

print(f" gold.dim_car      : {dim_car.count():,} rows, {len(dim_car.columns)} columns")


## dim_region
`silver.customers` + `silver.sales`

In [0]:
from pyspark.sql.functions import col, current_timestamp, monotonically_increasing_id

customers = spark.table("primeinsurance.silver.customers")
sales     = spark.table("primeinsurance.silver.sales")

region_customers = customers.filter(col("region").isNotNull())     .select(col("region").alias("region_name"))

region_sales = sales.filter(col("Region").isNotNull())     .select(col("Region").alias("region_name"))

dim_region = (
    region_customers
    .union(region_sales)
    .distinct()
    .orderBy("region_name")
    .withColumn("region_id", monotonically_increasing_id().cast("integer") + 1)
    .withColumn("gold_created_at", current_timestamp())
    .select(
        "region_id",
        "region_name"
    )
)

(
    dim_region.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.dim_region")
)

print(f" gold.dim_region   : {dim_region.count():,} rows, {len(dim_region.columns)} columns")


## dim_date
`silver.sales` — `ad_placed_on` + `sold_on`

> Claims dates excluded — all corrupted (`27:00.0`) across 1,000 rows

In [0]:
from pyspark.sql.functions import (
    col, to_date, to_timestamp, year, month, dayofmonth,
    dayofweek, quarter, date_format,
    current_timestamp, monotonically_increasing_id
)

sales = spark.table("primeinsurance.silver.sales")

# Extract date part from both timestamp columns
dates_ad = sales.filter(col("ad_placed_on").isNotNull()) \
    .select(to_date(col("ad_placed_on"), "dd-MM-yyyy HH:mm").alias("full_date"))

dates_sold = sales.filter(col("sold_on").isNotNull()) \
    .select(to_date(col("sold_on"), "dd-MM-yyyy HH:mm").alias("full_date"))

# Union → distinct → build all date attributes
dim_date = (
    dates_ad
    .union(dates_sold)
    .distinct()
    .filter(col("full_date").isNotNull())
    .orderBy("full_date")
    .withColumn("date_id",monotonically_increasing_id().cast("integer") + 1)
    .withColumn("year",year(col("full_date")))
    .withColumn("quarter",quarter(col("full_date")))
    .withColumn("month",month(col("full_date")))
    .withColumn("month_name",date_format(col("full_date"), "MMMM"))
    .withColumn("day",dayofmonth(col("full_date")))
    .withColumn("day_of_week",dayofweek(col("full_date")))
    .withColumn("day_name",date_format(col("full_date"), "EEEE"))
    .withColumn("gold_created_at",current_timestamp())
    .select(
        "date_id",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name"
    )
)

(
    dim_date.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.dim_date")
)

print(f" gold.dim_date : {dim_date.count():,} rows, {len(dim_date.columns)} columns")

# Preview
spark.table("primeinsurance.gold.dim_date").display()

## Preview — All 7 Tables

In [0]:
tables = [
    "fact_claims",
    "fact_sales",
    "dim_customer",
    "dim_policy",
    "dim_car",
    "dim_region",
    "dim_date"
]

for tbl in tables:
    print(f"\n▶ gold.{tbl}")
    spark.table(f"primeinsurance.gold.{tbl}").display()


## Summary

In [0]:
from pyspark.sql.functions import col

tables = [
    "fact_claims",
    "fact_sales",
    "dim_customer",
    "dim_policy",
    "dim_car",
    "dim_region",
    "dim_date"
]

print("=" * 50)
print(f"{'Table':<20} {'Rows':>8} {'Columns':>8}")
print("=" * 50)
for tbl in tables:
    df = spark.table(f"primeinsurance.gold.{tbl}")
    print(f"{tbl:<20} {df.count():>8,} {len(df.columns):>8}")
print("=" * 50)
print("\n Gold layer complete — primeinsurance.gold")
